In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import eodgdl
from eodgdl import giro
import informal_jobs_model as ijm
import pandas as pd

/Users/gperaza/Research/informal-jobs-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Economic sector of OD workers

Most OD workers did not report the activity of their employer (`giro_empresa`). The imputation model lives in the `eodgdl` package (`eodgdl.giro`, extra `eodgdl[giro]`): trained within the survey on the workers with an observed giro, from raw survey columns, the work-trip destination and mode and the destination's DENUE establishment mix, it predicts the survey's five native levels (Comercio, Servicio, Educación, Industria, Gobierno/sector público) and returns the full probability vector `prob_giro_<slug>`. Its training, validation, calibration and covariate-shift diagnostics are in `notebooks/giro_model.ipynb` of that repository.

This stage scores the survey with the fitted bundle and collapses the five giro probabilities to the four harmonized sector classes of the informality model (`sector.yaml` `od_giro`: Servicio and Educación → `servicios_transporte`, Industria → `manufactura_construccion`, Gobierno → `gobierno_otro_agricultura`; a many-to-one map, so the collapse is lossless for the marginalization over sectors).

In [2]:
tables = eodgdl.load_eod()
bundle = giro.load_model()  # fetched from the eodgdl data mirror (or $EODGDL_DATA_DIR)
print("giro bundle:", {k: v for k, v in bundle["metadata"].items() if k in ("eodgdl_version", "sklearn_version", "denue_release")})
print("selected models:", {k: v["model"] for k, v in bundle["metadata"]["selected"].items()})

od = giro.build_worker_features(tables)
od_giro = giro.impute_giro(bundle["model_with_education"], bundle["model_without_education"], od, with_education_features=bundle["features_with_education"], without_education_features=bundle["features_without_education"], destination_models=bundle["destination_models"])
print(f"OD workers: {len(od_giro):,}; imputed giro: {od_giro['giro_fue_imputado'].sum():,}; maximum probability-sum error: {giro.validate_probability_rows(od_giro):.3e}")
display(giro.calculate_model_usage(od_giro).round(4))
print("Giro distribution, all workers (weighted, %):")
print((giro.calculate_probabilistic_distribution(od_giro).set_index("giro")["weighted_share"] * 100).round(2).to_string())

giro bundle: {'denue_release': '202211', 'sklearn_version': '1.9.0', 'eodgdl_version': '0.2.0.dev0 (giro-model branch)'}
selected models: {'with_education': 'GradientBoosting', 'without_education': 'GradientBoosting'}


OD workers: 26,913; imputed giro: 9,484; maximum probability-sum error: 3.331e-16


,giro_model_used,sample_workers,weighted_population,sample_share,weighted_share
0,with_education,4962,366704,0.5232,0.5346
1,without_education,4522,319242,0.4768,0.4654


Giro distribution, all workers (weighted, %):
giro
comercio     30.79
servicio     35.07
educacion     1.50
industria    28.04
gobierno      4.60


## Sensitivity scenarios

The model is validated on known-giro households but applied to the unknown-giro workers, who differ. Two scenarios from `eodgdl.giro` are propagated to the informality headline in notebook 05: a refit of the selected models on known-giro rows reweighted to the unknown-giro profile, and a delta adjustment of the rare `gobierno` class to its observed share.

In [3]:
od_giro_shift, shift_diagnostics = giro.impute_under_covariate_shift(bundle["model_with_education"], bundle["model_without_education"], od, with_education_features=bundle["features_with_education"], without_education_features=bundle["features_without_education"])
od_giro_delta, delta_factor = giro.adjust_imputed_share(od_giro, "gobierno")
print("Shift-weighting diagnostics:"); print(shift_diagnostics.round(3).to_string())
print(f"\nDelta adjustment factor for gobierno among imputed workers: {delta_factor:.3f}")
imputed_shares = lambda frame: giro.calculate_probabilistic_distribution(frame[frame["giro_fue_imputado"]]).set_index("giro")["weighted_share"]
sensitivity = pd.DataFrame({"observed (known giro)": giro.calculate_distribution(od[~od["giro_desconocido"]], giro_column="giro").set_index("giro")["weighted_share"], "imputed: current": imputed_shares(od_giro), "imputed: shift-weighted": imputed_shares(od_giro_shift), "imputed: delta-adjusted": imputed_shares(od_giro_delta)}).reindex(giro.GIRO_CLASSES)
(sensitivity * 100).round(2)

Shift-weighting diagnostics:
rows                       17429.000
effective_sample_size       6820.462
top_decile_weight_share        0.190
odds_ratio_median              0.977
odds_ratio_p90                 2.250

Delta adjustment factor for gobierno among imputed workers: 0.693


,observed (known giro),imputed: current,imputed: shift-weighted,imputed: delta-adjusted
giro,,,,
comercio,33.36,24.84,24.03,25.15
servicio,34.84,35.61,34.88,36.64
educacion,1.59,1.31,1.21,1.38
industria,26.16,32.39,33.71,32.77
gobierno,4.05,5.85,6.18,4.05


## Outputs

`od_giro_imputed.parquet` keeps the giro model's own output; `od_sector_imputed.parquet` is `od_harmonized` joined with that output collapsed to the harmonized sector classes (`ijm.attach_sector_probabilities`), the input of the informality stage; the sensitivity scenarios are collapsed the same way.

In [4]:
output_directory = ROOT / "outputs"
od_giro[giro.OUTPUT_COLUMNS].to_parquet(output_directory / "od_giro_imputed.parquet", index=False)

od_harmonized = pd.read_parquet(output_directory / "od_harmonized.parquet")
od_sector_imputed = ijm.attach_sector_probabilities(od_harmonized, od_giro[giro.OUTPUT_COLUMNS])
od_sector_imputed.to_parquet(output_directory / "od_sector_imputed.parquet", index=False)
sensitivity_output = pd.concat([ijm.attach_sector_probabilities(od_harmonized, frame[giro.OUTPUT_COLUMNS]).assign(scenario=name) for name, frame in (("shift_weighted", od_giro_shift), ("delta_adjusted", od_giro_delta))], ignore_index=True)
sensitivity_output.to_parquet(output_directory / "od_sector_imputed_sensitivity.parquet", index=False)

sector_columns = [f"prob_sector_{s}" for s in ijm.SECTOR_CLASSES]
print("Sector distribution after collapsing to the harmonized classes (weighted, %):")
print(((od_sector_imputed[sector_columns].multiply(od_sector_imputed["expansion_factor"], axis=0).sum() / od_sector_imputed["expansion_factor"].sum()) * 100).round(2).to_string())

Sector distribution after collapsing to the harmonized classes (weighted, %):
prob_sector_comercio                     30.79
prob_sector_gobierno_otro_agricultura      4.6
prob_sector_manufactura_construccion     28.04
prob_sector_servicios_transporte         36.57
